In [42]:
import gymnasium as gym
import json
from mountainCar_neural import neural_agent
import torch
import os

In [2]:
env = gym.make("MountainCar-v0")

In [57]:
def run_experiment(tag="opt"):
    ret_list = []
    act_list = []
    """
    return: list of returned env step
    """
    count = 0
    state = env.reset()
    end = False
    action = env.action_space.sample()
    ret_list.append(state[0].tolist())
    act_list.append(action.item())
    while not end:
        ret = env.step(action)
        ret_list.append(ret[0].tolist())
        if ret[0][0] < 0 and ret[0][1] < 0:
            action = 0 if tag == "opt" else 2
        elif ret[0][0] < 0 and ret[0][1] > 0:
            action = 2 if tag == "opt" else 2
        elif ret[0][0] > 0 and ret[0][1] > 0:
            action = 2 if tag == "opt" else 2
        elif ret[0][0] > 0 and ret[0][1] < 0:
            action = 0 if tag == "opt" else 2
        act_list.append(action)
        end = ret[2] or ret[3]
        count += 1
    env.close()
    return ret_list, act_list

In [4]:
# data generation for opt
ret_list_sum=[]
act_list_sum=[] 
for i in range(2):
 ret, act= run_experiment()
 ret_list_sum.extend(ret)
 act_list_sum.extend(act)

In [60]:
# data generation for bad
ret_list_sum = []
act_list_sum = []
for i in range(5):
    ret, act = run_experiment("bad")
    ret_list_sum.extend(ret)
    act_list_sum.extend(act)

In [61]:
with open("opt_data.json", "w") as f:
    json.dump({
        'state':ret_list_sum,
        'action':act_list_sum
    },f)

In [10]:
env.close()

In [68]:
# expose learned weight
log_dir = 'log/opt/checkpoint/9.pkl'
# log_dir = 'log/bad2/checkpoint/9.pkl'
net=neural_agent()

In [69]:
net.load_state_dict(
    torch.load(
        log_dir,
        map_location=torch.device('cpu'),
    )
)

<All keys matched successfully>

In [70]:
net.state_dict()

OrderedDict([('paramA', tensor(0.0158)),
             ('paramB', tensor(2.0039)),
             ('paramC', tensor(-0.0034)),
             ('paramD', tensor(2.0047))])